# Saving 
My fianl goal is to create a table where each row is an user id, and I have columns such as:
1) Num like day one. 
2) Num posts day two.
3) Num of blocks day three. 
4) Joining date.

# Plan
1. Consider just the profiles that are present in chunk_0_posts_parquet: "user of interests". From them create a dictionary for fast joining_date_lookups.
2. "posts" time filtering of two weeks. 
3. We can further reduce the users of interest by asking if it has at least two posts. 
3. "blocks", "follows", "likes" filtering based on:
   1. The event happened in the first week after the joining date. 
   2. The user is in the user of interests. 

NB: We should create our final dataset consider the posts in chunks.

In [9]:
# Load the posts data
posts_path = "../data/posting/cleaned/chunk_0_posts.parquet"
posts_table = pq.read_table(posts_path)

posts_df = posts_table.to_pandas()

print(f"Initial posts count: {len(posts_df)}")
print(f"Unique users: {posts_df['did_id'].nunique()}")

user_post_counts = posts_df.groupby('did_id').size().reset_index(name='post_count')

# Filter out users with less than MIN_POSTS_PER_USER posts.
active_posters = user_post_counts[user_post_counts['post_count'] >= MIN_POSTS_PER_USER]['did_id'].values
filtered_posts_df = posts_df[posts_df['did_id'].isin(active_posters)]

print(f"\nPosts after filtering users with <{MIN_POSTS_PER_USER} posts: {len(filtered_posts_df)}")
print(f"Active posters (≥{MIN_POSTS_PER_USER} posts): {len(active_posters)}")

Initial posts count: 4994663
Unique users: 130111

Posts after filtering users with <2 posts: 4957527
Active users (≥2 posts): 92975

Posts after filtering users with <2 posts: 4957527
Active users (≥2 posts): 92975


In [15]:
# Create a fast lookup dictionary: did_id -> join_date
join_date_dict = dict(zip(
    user_of_interests['did_id'],
    user_of_interests['created_at']
))

print(f"Join date lookup dictionary created: {len(join_date_dict)} entries")

Join date lookup dictionary created: 65376 entries


In [23]:
# Generic function to filter event databases with multithreading
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

def filter_events(input_path, output_path, db_name, is_valid_event, num_threads=4):
    """
    Filter event databases by row groups with custom validation logic (multithreaded).
    
    Args:
        input_path (str): Path to input parquet file
        output_path (str): Path to output parquet file
        db_name (str): Name of database (for logging)
        is_valid_event (callable): Function that takes a row and returns True/False
        num_threads (int): Number of threads to use for processing row groups
    """
    print(f"Filtering {db_name} by row groups (using {num_threads} threads)...")
    
    # Open Parquet file
    pf = pq.ParquetFile(input_path)
    print(f"Total {db_name}: {pf.metadata.num_rows:,}")
    print(f"Row groups: {pf.num_row_groups}")
    
    # Get input file size
    input_size = os.path.getsize(input_path)
    print(f"Input file size: {input_size / (1024**3):.2f} GB")
    
    # Process row groups and collect filtered tables
    filtered_tables = {}
    total_rows_processed = 0
    total_rows_kept = 0
    
    def process_row_group(rg_index):
        """Process a single row group and return (index, filtered_table, rows_kept)"""
        row_group_table = pf.read_row_group(rg_index)
        rows_in_group = row_group_table.num_rows
        
        # Convert to pandas for easier filtering with custom logic
        row_group_df = row_group_table.to_pandas()
        
        # Apply custom validation function
        filtered_df = row_group_df[row_group_df.apply(is_valid_event, axis=1)]
        
        # Convert back to PyArrow table
        filtered_table = pa.Table.from_pandas(filtered_df, preserve_index=False)
        
        return rg_index, filtered_table, len(filtered_df), rows_in_group
    
    # Use ThreadPoolExecutor to process row groups in parallel
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = {executor.submit(process_row_group, i): i for i in range(pf.num_row_groups)}
        
        for future in as_completed(futures):
            rg_index, filtered_table, rows_kept, rows_total = future.result()
            filtered_tables[rg_index] = filtered_table
            total_rows_processed += rows_total
            total_rows_kept += rows_kept
            
            print(f"✅ Row group {rg_index+1}/{pf.num_row_groups}: {rows_total:,} → {rows_kept:,} rows")
    
    # Write all filtered tables in order
    print("\nWriting filtered results...")
    writer = None
    for i in range(pf.num_row_groups):
        filtered_table = filtered_tables[i]
        
        # Initialize writer with first filtered table schema
        if writer is None:
            writer = pq.ParquetWriter(
                output_path,
                filtered_table.schema,
                compression='zstd',
                use_dictionary=True,
                write_statistics=True
            )
        
        # Write filtered row group
        if filtered_table.num_rows > 0:
            writer.write_table(filtered_table)
    
    if writer:
        writer.close()
    
    # Calculate statistics
    retention_rate = (total_rows_kept / total_rows_processed * 100) if total_rows_processed > 0 else 0
    output_size = os.path.getsize(output_path)
    size_reduction = input_size - output_size
    size_reduction_percent = (size_reduction / input_size * 100) if input_size > 0 else 0
    
    print("\n" + "="*60)
    print(f"{db_name.upper()} FILTERING SUMMARY:")
    print("="*60)
    print(f"Row groups processed: {pf.num_row_groups} (with {num_threads} threads)")
    print(f"Rows:       {total_rows_processed:,} → {total_rows_kept:,} ({retention_rate:.1f}% kept)")
    print(f"File size:  {input_size / (1024**3):.2f} GB → {output_size / (1024**3):.2f} GB")
    print(f"Reduction:  {size_reduction / (1024**3):.2f} GB ({size_reduction_percent:.1f}%)")
    print(f"Output: {output_path}")
    print("="*60)

In [25]:
# Load the filtered blocks
blocks_table = pq.read_table(blocks_output_path)
blocks_df = blocks_table.to_pandas()

print(f"Filtered blocks loaded: {len(blocks_df)}")
print("\nBlocks schema:")
print(blocks_table.schema)
print("\nFirst few rows:")
print(blocks_df.head())

Filtered blocks loaded: 265695

Blocks schema:
created_at: timestamp[us, tz=UTC]
did_id: int64
subject_id: int64
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 465

First few rows:
                        created_at  did_id  subject_id
0 2024-11-16 15:16:16.539000+00:00     180     3640019
1 2024-11-25 13:34:08.890000+00:00     219       51454
2 2024-11-28 08:42:33.595000+00:00     219    15318020
3 2025-01-19 22:57:17.917000+00:00     219     1664267
4 2025-01-22 22:04:05.166000+00:00     219    32226456


In [4]:
# Inner join profiles with posts
merged_df = filtered_posts_df.merge(
    active_profiles, on='did_id', how='inner'
)
print(merged_df.head())

                        created_at  did_id                        join_date
0 2024-08-30 21:25:28.002000+00:00     318 2024-08-30 21:04:26.303000+00:00
1 2024-08-30 22:29:13.143000+00:00     318 2024-08-30 21:04:26.303000+00:00
2 2024-08-31 14:01:03.295000+00:00     318 2024-08-30 21:04:26.303000+00:00
3 2024-08-31 14:23:46.432000+00:00     318 2024-08-30 21:04:26.303000+00:00
4 2024-08-31 14:39:12.549000+00:00     318 2024-08-30 21:04:26.303000+00:00


In [ ]:
# Calculate days since joining, filter the first month of activity, merge them to user_table

merged_df['days_since_join'] = (
    (merged_df['created_at'] - merged_df['join_date']).dt.total_seconds() / (24 * 3600)
).round().astype(int)

first_month_posts = merged_df[
    (merged_df['days_since_join'] >= 0) & 
    (merged_df['days_since_join'] <= 15)
]

daily_post_counts = first_month_posts.groupby(['did_id', 'days_since_join']).size().reset_index(name='post_count')

# Pivot
time_series_wide = daily_post_counts.pivot_table(
    index='did_id', 
    columns='days_since_join', 
    values='post_count', 
    fill_value=0
).reset_index()

# Rename the day-posts columns
time_series_wide.columns = ['did_id'] + [f'day_{int(col)}_posts' for col in time_series_wide.columns[1:]]

# Merge into final table
user_table = user_table.merge(time_series_wide, on='did_id', how='left')

print(user_table.head())

   did_id                        join_date                  first_post_date  \
0     318 2024-08-30 21:04:26.303000+00:00 2024-08-30 21:25:28.002000+00:00   
1    1032 2024-10-19 12:08:26.894000+00:00 2024-11-17 23:04:06.860000+00:00   
2    1738 2024-11-24 22:41:30.245000+00:00 2024-11-24 22:44:22.029000+00:00   
3    3316 2024-06-08 03:54:37.834000+00:00 2024-06-08 06:02:30.972000+00:00   
4    3360 2024-11-17 05:41:26.862000+00:00 2024-11-17 05:44:20.897000+00:00   

   day_0_posts  day_1_posts  day_2_posts  day_3_posts  day_4_posts  \
0          2.0          5.0          4.0          4.0          3.0   
1          0.0          0.0          0.0          0.0          0.0   
2         19.0          0.0          0.0          0.0          0.0   
3          5.0          0.0          0.0          1.0          0.0   
4          1.0         18.0         11.0          4.0         20.0   

   day_5_posts  day_6_posts  ...  day_21_posts  day_22_posts  day_23_posts  \
0          3.0          0.

In [7]:
# Save the processed data for future use
output_path = "../data/posting/processed/user_activity.parquet"
user_table.to_parquet(output_path, index=False)
print(f"\nData saved to: {output_path}")


Data saved to: ../data/posting/processed/user_activity.parquet
